In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = Path("data/processed/cleaned/cleaned_data.csv")
LAB11_DIR = Path("data/processed/analytics/lab11")
LAB11_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Cleaned data exists:", DATA_PATH.exists())
print("Lab 11 output directory:", LAB11_DIR)

Project root: c:\Users\38760\Desktop\Unstructured_Data\News_Media_Monitoring_Pipeline_UD
Cleaned data exists: True
Lab 11 output directory: data\processed\analytics\lab11


In [2]:
from src.embeddings.embedder import (
    build_news_text,
    build_texts_from_dataframe,
    embed_texts,
    embed_single,
    cosine_similarity,
    dot_product,
    euclidean_distance,
    cosine_similarity_matrix,
    rank_texts_by_similarity,
    compare_similarity_measures,
)

from src.embeddings.chroma_store import (
    get_chroma_client,
    get_collection,
    add_news_to_collection,
    query_collection,
    query_collection_multi,
    chroma_results_to_dataframe,
)

from src.embeddings.search_engine import (
    keyword_search,
    semantic_search,
    compare_search,
    compare_synonym_queries,
)

from src.embeddings.hybrid_search import (
    hybrid_search,
    compare_all_search_methods,
)

c:\Users\38760\Desktop\Unstructured_Data\News_Media_Monitoring_Pipeline_UD\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the cleaned news dataset

Lab 11 continues from Lab 9 and Lab 10, so the input is the cleaned news dataset.

In [3]:
df = pd.read_csv(DATA_PATH, low_memory=False)

print("Cleaned dataset shape:", df.shape)
display(df[["record_id", "title", "content_text", "category", "document_type", "language", "rating_score"]].head())

Cleaned dataset shape: (1318, 35)


,record_id,title,content_text,category,document_type,language,rating_score
0,52,Election Update,Election Update,politics,excel,unknown,4.0
1,53,AI Market Growth,AI Market Growth,business,excel,unknown,7.0
2,54,Sports Highlights,Sports Highlights,sports,excel,unknown,5.0
3,55,Climate Policy Shift,Climate Policy Shift,politics,excel,unknown,3.0
4,56,Café Culture Trends,Café Culture Trends,culture,excel,unknown,6.0


## 2. Generate sample embeddings

The model `all-MiniLM-L6-v2` converts each text into a 384-dimensional vector. Similar texts should have vectors that are closer together.

In [4]:
sample_texts = [
    "AI companies expand semiconductor partnerships and cloud infrastructure",
    "Technology firms invest in artificial intelligence infrastructure",
    "Local sports team wins the championship final",
    "Climate policy changes affect energy markets and government regulation",
    "Stock markets rise as oil prices fall and investors watch central banks",
]

sample_embeddings = embed_texts(sample_texts, batch_size=5, normalize=True)

print("Embedding shape:", sample_embeddings.shape)
print("Embedding dimension:", sample_embeddings.shape[1])

Loading model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 21562.93it/s]


Model loaded successfully
Embedding shape: (5, 384)
Embedding dimension: 384


In [5]:
similarity_matrix = cosine_similarity_matrix(sample_embeddings)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=["AI chips", "AI infrastructure", "Sports", "Climate energy", "Stocks oil"],
    columns=["AI chips", "AI infrastructure", "Sports", "Climate energy", "Stocks oil"],
)

display(similarity_df.round(3))

,AI chips,AI infrastructure,Sports,Climate energy,Stocks oil
AI chips,1.000,0.637,-0.025,0.126,0.133
AI infrastructure,0.637,1.000,0.035,0.192,0.193
Sports,-0.025,0.035,1.000,-0.011,-0.089
Climate energy,0.126,0.192,-0.011,1.000,0.327
Stocks oil,0.133,0.193,-0.089,0.327,1.000


The AI-related texts should be more similar to each other than to the sports text. This demonstrates that embeddings capture meaning, not only exact keywords.

## 3. Compare cosine similarity, dot product, and Euclidean distance

Cosine similarity and dot product are higher for more similar normalized embeddings. Euclidean distance is lower for more similar embeddings.

In [6]:
comparison_df = compare_similarity_measures(
    "AI companies expand semiconductor partnerships and cloud infrastructure",
    "Technology firms invest in artificial intelligence infrastructure",
    "Local sports team wins the championship final",
)

display(comparison_df.round(4))

,pair,cosine_similarity,dot_product,euclidean_distance
0,A vs B,0.6366,0.6366,0.8525
1,A vs C,-0.0254,-0.0254,1.4320


In [7]:
query = "artificial intelligence technology market"

ranked_sample_results = rank_texts_by_similarity(
    query=query,
    texts=sample_texts,
    embeddings=sample_embeddings,
    top_k=5,
)

print("Query:", query)
display(ranked_sample_results)

Query: artificial intelligence technology market


,rank,text_index,similarity_score,text
0,1,1,0.763181,Technology firms invest in artificial intellig...
1,2,0,0.496310,AI companies expand semiconductor partnerships...
2,3,4,0.197928,Stock markets rise as oil prices fall and inve...
3,4,3,0.128334,Climate policy changes affect energy markets a...
4,5,2,-0.000546,Local sports team wins the championship final


## 4. Build searchable text from the real dataset

For each news record, the search text combines title, content, category, document type, source, and language.

In [8]:
preview_texts = build_texts_from_dataframe(df.head(3))

for i, text in enumerate(preview_texts, start=1):
    print(f"Document {i}:")
    print(text[:600])
    print()

Document 1:
Title: Election Update | Content: Election Update | Category: politics | Document type: excel | Source: bbc | Language: unknown

Document 2:
Title: AI Market Growth | Content: AI Market Growth | Category: business | Document type: excel | Source: reuters | Language: unknown

Document 3:
Title: Sports Highlights | Content: Sports Highlights | Category: sports | Document type: excel | Source: cnn | Language: unknown



## 5. Set up ChromaDB and populate the persistent collection

The collection is named `data` and is stored on disk in `data/embeddings/chroma_db/`. Re-running this cell skips records already stored in the collection.

In [9]:
client = get_chroma_client()
collection = get_collection(client, reset=False)

total_records = add_news_to_collection(
    df,
    collection,
    batch_size=100,
)

print("ChromaDB collection name:", collection.name)
print("Total records in ChromaDB:", total_records)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10620.06it/s]


Collection already has 1318 records
Added 0 records
Skipped 1318 records
Collection now contains 1318 records
ChromaDB collection name: data
Total records in ChromaDB: 1318


## 6. Semantic search with ChromaDB

Semantic search returns records based on meaning, even when the query words do not exactly match the article text.

In [10]:
semantic_query = "artificial intelligence business technology market"

semantic_result = query_collection(
    collection,
    semantic_query,
    n_results=5,
)

semantic_df = chroma_results_to_dataframe(semantic_result)

display(semantic_df[["rank", "title", "category", "document_type", "year", "rating_score", "similarity_score"]])

,rank,title,category,document_type,year,rating_score,similarity_score
0,1,AI Market Growth,business,excel,2026,7.00,0.547831
1,2,"Commentary: AI drives global tech layoffs, but...",json,json,0,2.60,0.462358
2,3,"Global Market | Nikkei jumps nearly 2% as AI, ...",news_api,news_api,0,2.60,0.378731
3,4,SoftServe Launches New Brand Identity: ‘Techno...,json,json,0,2.60,0.373444
4,5,Gen Z workers are so fearful AI will take thei...,json,json,0,1.41,0.362031


## 7. Multi-query ChromaDB search

ChromaDB can handle multiple semantic queries in one call.

In [11]:
multi_queries = [
    "artificial intelligence and technology markets",
    "climate policy and energy",
    "sports championship results",
]

multi_result = query_collection_multi(
    collection,
    multi_queries,
    n_results=3,
)

multi_query_rows = []

for query_index, query_text in enumerate(multi_queries):
    result_df = chroma_results_to_dataframe(multi_result, query_index=query_index)
    result_df["query"] = query_text
    multi_query_rows.append(result_df)

multi_query_df = pd.concat(multi_query_rows, ignore_index=True)

display(multi_query_df[["query", "rank", "title", "category", "similarity_score"]])
multi_query_df.to_csv(LAB11_DIR / "multi_query_semantic_results.csv", index=False)

,query,rank,title,category,similarity_score
0,artificial intelligence and technology markets,1,AI Market Growth,business,0.514088
1,artificial intelligence and technology markets,2,"Commentary: AI drives global tech layoffs, but...",json,0.469032
2,artificial intelligence and technology markets,3,"Global Market | Nikkei jumps nearly 2% as AI, ...",news_api,0.438687
3,climate policy and energy,1,Climate Policy Shift,politics,0.497206
4,climate policy and energy,2,A New Study Challenges the Climate Establishme...,news_api,0.470571
5,climate policy and energy,3,India no longer interested in hosting COP33 cl...,json,0.367867
6,sports championship results,1,Weekend Match Buzz,sports,0.436989
7,sports championship results,2,Sports Highlights,sports,0.414322
8,sports championship results,3,NZ Dairy Industry Awards 2026: Olivia Sutton a...,news_api,0.256896


## 8. Metadata filtering

This section demonstrates ChromaDB filters using `$eq`, `$gte`, `$in`, `$and`, and `$or`.

In [12]:
filter_examples = [
    ("$eq category business", {"category": {"$eq": "business"}}),
    ("$gte year 2026", {"year": {"$gte": 2026}}),
    ("$in selected categories", {"category": {"$in": ["business", "technology", "politics"]}}),
    (
        "$and business and rating >= 5",
        {
            "$and": [
                {"category": {"$eq": "business"}},
                {"rating_score": {"$gte": 5}},
            ]
        },
    ),
    (
        "$or climate or sports",
        {
            "$or": [
                {"category": {"$eq": "politics"}},
                {"category": {"$eq": "sports"}},
            ]
        },
    ),
]

filter_rows = []

for filter_name, where_filter in filter_examples:
    result = query_collection(
        collection,
        "artificial intelligence business technology market",
        n_results=5,
        where=where_filter,
    )
    result_df = chroma_results_to_dataframe(result)
    result_df["filter_name"] = filter_name
    filter_rows.append(result_df)

filter_results_df = pd.concat(filter_rows, ignore_index=True)

display(filter_results_df[["filter_name", "rank", "title", "category", "year", "rating_score", "similarity_score"]])
filter_results_df.to_csv(LAB11_DIR / "metadata_filter_results.csv", index=False)

,filter_name,rank,title,category,year,rating_score,similarity_score
0,$eq category business,1,AI Market Growth,business,2026,7.0,0.547831
1,$eq category business,2,Streaming Rights Watch,business,2026,4.0,0.254049
2,$eq category business,3,Inflation Briefing,business,2026,2.0,0.177555
3,$gte year 2026,1,AI Market Growth,business,2026,7.0,0.547831
4,$gte year 2026,2,Streaming Rights Watch,business,2026,4.0,0.254049
5,$gte year 2026,3,München Startup Round,technology,2026,8.0,0.188381
6,$gte year 2026,4,Inflation Briefing,business,2026,2.0,0.177555
7,$gte year 2026,5,Café Culture Trends,culture,2026,6.0,0.163083
8,$in selected categories,1,AI Market Growth,business,2026,7.0,0.547831
9,$in selected categories,2,Streaming Rights Watch,business,2026,4.0,0.254049


## 9. Keyword search vs semantic search

Keyword search matches exact words. Semantic search finds related meaning. The comparison shows where each method succeeds.

In [13]:
query = "artificial intelligence technology market"

keyword_df = keyword_search(
    query=query,
    df=df,
    n_results=5,
)

semantic_df = semantic_search(
    query=query,
    n_results=5,
    collection=collection,
)

print("Keyword search results:")
display(keyword_df)

print("Semantic search results:")
display(semantic_df[["rank", "title", "category", "document_type", "year", "similarity_score"]])

Keyword search results:


,record_id,title,category,document_type,published_year,language,rating_score,keyword_score,phrase_match,search_type
0,17,"Global Market | Nikkei jumps nearly 2% as AI, ...",news_api,news_api,NaN,unknown,2.60,2,False,keyword
1,9,Nvidia preparing Groq chips that can be sold i...,news_api,news_api,NaN,unknown,2.28,2,False,keyword
2,46,Untitled record,pdf,pdf,NaN,unknown,10.00,1,False,keyword
3,78,Untitled record,pdf,pdf,NaN,unknown,10.00,1,False,keyword
4,110,Untitled record,pdf,pdf,NaN,unknown,10.00,1,False,keyword


Semantic search results:


,rank,title,category,document_type,year,similarity_score
0,1,"Commentary: AI drives global tech layoffs, but...",json,json,0,0.499213
1,2,AI Market Growth,business,excel,2026,0.485241
2,3,"Global Market | Nikkei jumps nearly 2% as AI, ...",news_api,news_api,0,0.403436
3,4,Gen Z workers are so fearful AI will take thei...,json,json,0,0.388903
4,5,US State Dept orders global warning about alle...,json,json,0,0.384712


In [14]:
comparison = compare_search(
    query="artificial intelligence technology market",
    df=df,
    collection=collection,
    n_results=5,
)

comparison["keyword"].to_csv(LAB11_DIR / "keyword_search_results_notebook.csv", index=False)
comparison["semantic"].to_csv(LAB11_DIR / "semantic_search_results_notebook.csv", index=False)

--- Query: 'artificial intelligence technology market' ---

Keyword search found 5 results:
  Global Market | Nikkei jumps nearly 2% as AI, chip stocks rally (nan) - news_api
  Nvidia preparing Groq chips that can be sold in Chinese market, sources say (nan) - news_api
  Untitled record (nan) - pdf
  Untitled record (nan) - pdf
  Untitled record (nan) - pdf

Semantic search found 5 results:
  [0.499] Commentary: AI drives global tech layoffs, but Intel, QNAP, and Realtek tell different stories (0) - json
  [0.485] AI Market Growth (2026) - business
  [0.403] Global Market | Nikkei jumps nearly 2% as AI, chip stocks rally (0) - news_api
  [0.389] Gen Z workers are so fearful AI will take their job they’re intentionally sabotaging their company’s AI rollout (0) - json
  [0.385] US State Dept orders global warning about alleged AI thefts by DeepSeek, other Chinese firms (0) - json


Keyword search is useful for exact names, titles, IDs, or terms. Semantic search is better when the user describes a concept using synonyms or paraphrased wording.

## 10. Synonym query overlap

This measures whether two similar queries return overlapping results. A higher overlap means the method is more consistent for synonyms and paraphrases.

In [15]:
synonym_pairs = [
    ("AI market", "artificial intelligence business"),
    ("climate policy", "environment regulation energy"),
    ("sports championship", "match results tournament"),
]

overlap_rows = []

for query_a, query_b in synonym_pairs:
    overlap_df = compare_synonym_queries(
        query_a=query_a,
        query_b=query_b,
        df=df,
        collection=collection,
        n_results=10,
    )
    overlap_rows.append(overlap_df)

overlap_report = pd.concat(overlap_rows, ignore_index=True)

display(overlap_report)
overlap_report.to_csv(LAB11_DIR / "synonym_overlap_report_notebook.csv", index=False)

,method,query_a,query_b,overlap_count,overlap_ratio,set_a_size,set_b_size
0,keyword,AI market,artificial intelligence business,1,0.1,10,10
1,semantic,AI market,artificial intelligence business,7,0.7,10,10
2,keyword,climate policy,environment regulation energy,0,0.0,10,4
3,semantic,climate policy,environment regulation energy,7,0.7,10,10
4,keyword,sports championship,match results tournament,5,0.5,10,10
5,semantic,sports championship,match results tournament,10,1.0,10,10


In most cases, semantic search should have equal or higher overlap for synonym-style queries because it searches by meaning rather than exact word matches.

## 11. Hybrid search with Reciprocal Rank Fusion

Hybrid search combines keyword and semantic results using Reciprocal Rank Fusion:

`score = 1 / (k + rank)`

This gives good results when both exact matching and meaning-based retrieval are useful.

In [16]:
hybrid_df = hybrid_search(
    query="artificial intelligence technology market",
    df=df,
    collection=collection,
    n_results=10,
    k=60,
)

display(hybrid_df[["title", "category", "document_type", "rrf_score", "keyword_rank", "semantic_rank"]])
hybrid_df.to_csv(LAB11_DIR / "hybrid_search_results_notebook.csv", index=False)

,title,category,document_type,rrf_score,keyword_rank,semantic_rank
0,"Global Market | Nikkei jumps nearly 2% as AI, ...",news_api,news_api,0.032266,1.0,3.0
1,AI Market Growth,business,excel,0.029828,13.0,2.0
2,Nvidia preparing Groq chips that can be sold i...,news_api,news_api,0.028324,2.0,22.0
3,"Commentary: AI drives global tech layoffs, but...",json,json,0.027505,30.0,1.0
4,"GATE 2026 final answer key not released yet, r...",news_api,news_api,0.023810,24.0,24.0
5,Untitled record,pdf,pdf,0.015873,3.0,NaN
6,Untitled record,pdf,pdf,0.015625,4.0,NaN
7,Gen Z workers are so fearful AI will take thei...,json,json,0.015625,NaN,4.0
8,Untitled record,pdf,pdf,0.015385,5.0,NaN
9,US State Dept orders global warning about alle...,json,json,0.015385,NaN,5.0


In [17]:
all_methods = compare_all_search_methods(
    query="artificial intelligence technology market",
    df=df,
    collection=collection,
    n_results=5,
)

all_methods["keyword"].to_csv(LAB11_DIR / "keyword_results_compare_all.csv", index=False)
all_methods["semantic"].to_csv(LAB11_DIR / "semantic_results_compare_all.csv", index=False)
all_methods["hybrid"].to_csv(LAB11_DIR / "hybrid_results_compare_all.csv", index=False)

--- Query: 'artificial intelligence technology market' ---

Keyword results:
  Global Market | Nikkei jumps nearly 2% as AI, chip stocks rally - news_api
  Nvidia preparing Groq chips that can be sold in Chinese market, sources say - news_api
  Untitled record - pdf
  Untitled record - pdf
  Untitled record - pdf

Semantic results:
  [0.499] Commentary: AI drives global tech layoffs, but Intel, QNAP, and Realtek tell different stories - json
  [0.485] AI Market Growth - business
  [0.403] Global Market | Nikkei jumps nearly 2% as AI, chip stocks rally - news_api
  [0.389] Gen Z workers are so fearful AI will take their job they’re intentionally sabotaging their company’s AI rollout - json
  [0.385] US State Dept orders global warning about alleged AI thefts by DeepSeek, other Chinese firms - json

Hybrid results:
  [0.032266] Global Market | Nikkei jumps nearly 2% as AI, chip stocks rally - news_api
  [0.029828] AI Market Growth - business
  [0.016393] Commentary: AI drives global tech

## 12. Analytical search questions

The following questions demonstrate the search system end-to-end using semantic search and metadata filters.

In [20]:
questions = [
    {
        "question": "Which records discuss artificial intelligence, business, and technology?",
        "query": "artificial intelligence business technology market",
        "filter": None,
    },
    {
        "question": "Which business records are semantically related to AI and markets?",
        "query": "AI technology market growth",
        "filter": {"category": {"$eq": "business"}},
    },
    {
        "question": "Which 2026 records discuss climate, energy, or policy?",
        "query": "climate energy policy regulation",
        "filter": {"year": {"$gte": 2026}},
    },
]

analytical_rows = []

for q in questions:
    result = query_collection(
        collection,
        q["query"],
        n_results=3,
        where=q["filter"],
    )

    result_df = chroma_results_to_dataframe(result)

    if result_df.empty:
        analytical_rows.append(pd.DataFrame([{
            "question": q["question"],
            "query": q["query"],
            "rank": None,
            "title": "No results found",
            "category": None,
            "year": None,
            "similarity_score": None,
            "finding": "No matching records were returned for this query.",
        }]))
        continue

    result_df["question"] = q["question"]
    result_df["query"] = q["query"]

    top_title = result_df.iloc[0].get("title", "Unknown title")
    top_category = result_df.iloc[0].get("category", "unknown")
    top_score = result_df.iloc[0].get("similarity_score", 0)

    result_df["finding"] = (
        "Top result: "
        + str(top_title)
        + " | Category: "
        + str(top_category)
        + " | Similarity score: "
        + result_df["similarity_score"].round(4).astype(str)
    )

    analytical_rows.append(result_df)

analytical_results_df = pd.concat(analytical_rows, ignore_index=True)

output_cols = [
    "question",
    "query",
    "rank",
    "title",
    "category",
    "year",
    "rating_score",
    "similarity_score",
    "finding",
]

existing_output_cols = [col for col in output_cols if col in analytical_results_df.columns]

display(analytical_results_df[existing_output_cols])

analytical_results_df.to_csv(
    LAB11_DIR / "analytical_search_questions.csv",
    index=False,
)

print("Saved analytical questions to:", LAB11_DIR / "analytical_search_questions.csv")

,question,query,rank,title,category,year,rating_score,similarity_score,finding
0,"Which records discuss artificial intelligence,...",artificial intelligence business technology ma...,1,AI Market Growth,business,2026,7.0,0.547831,Top result: AI Market Growth | Category: busin...
1,"Which records discuss artificial intelligence,...",artificial intelligence business technology ma...,2,"Commentary: AI drives global tech layoffs, but...",json,0,2.6,0.462358,Top result: AI Market Growth | Category: busin...
2,"Which records discuss artificial intelligence,...",artificial intelligence business technology ma...,3,"Global Market | Nikkei jumps nearly 2% as AI, ...",news_api,0,2.6,0.378731,Top result: AI Market Growth | Category: busin...
3,Which business records are semantically relate...,AI technology market growth,1,AI Market Growth,business,2026,7.0,0.599471,Top result: AI Market Growth | Category: busin...
4,Which business records are semantically relate...,AI technology market growth,2,Streaming Rights Watch,business,2026,4.0,0.144076,Top result: AI Market Growth | Category: busin...
5,Which business records are semantically relate...,AI technology market growth,3,Inflation Briefing,business,2026,2.0,0.134612,Top result: AI Market Growth | Category: busin...
6,"Which 2026 records discuss climate, energy, or...",climate energy policy regulation,1,Climate Policy Shift,politics,2026,3.0,0.445453,Top result: Climate Policy Shift | Category: p...
7,"Which 2026 records discuss climate, energy, or...",climate energy policy regulation,2,Inflation Briefing,business,2026,2.0,0.129432,Top result: Climate Policy Shift | Category: p...
8,"Which 2026 records discuss climate, energy, or...",climate energy policy regulation,3,Café Culture Trends,culture,2026,6.0,0.049249,Top result: Climate Policy Shift | Category: p...


Saved analytical questions to: data\processed\analytics\lab11\analytical_search_questions.csv


### Analytical interpretation

1. The AI/business technology query should retrieve records about AI markets, technology companies, chips, or business/market activity.
2. The business metadata filter narrows semantic search to business records only, showing how vector search can be combined with structured filtering.
3. The climate/policy query with a year filter demonstrates end-to-end semantic search with metadata constraints.

## 13. Pipeline output verification

The Lab 11 pipeline integration saves search outputs into `data/processed/analytics/lab11/`.

In [21]:
expected_files = [
    LAB11_DIR / "keyword_search_results.csv",
    LAB11_DIR / "semantic_search_results.csv",
    LAB11_DIR / "hybrid_search_results.csv",
    LAB11_DIR / "synonym_query_overlap.csv",
    LAB11_DIR / "multi_query_semantic_results.csv",
    LAB11_DIR / "metadata_filter_results.csv",
    LAB11_DIR / "analytical_search_questions.csv",
]

verification_df = pd.DataFrame({
    "file": [str(path) for path in expected_files],
    "exists": [path.exists() for path in expected_files],
})

display(verification_df)

,file,exists
0,data\processed\analytics\lab11\keyword_search_...,True
1,data\processed\analytics\lab11\semantic_search...,True
2,data\processed\analytics\lab11\hybrid_search_r...,True
3,data\processed\analytics\lab11\synonym_query_o...,True
4,data\processed\analytics\lab11\multi_query_sem...,True
5,data\processed\analytics\lab11\metadata_filter...,True
6,data\processed\analytics\lab11\analytical_sear...,True


## Final Lab 11 conclusion

This notebook demonstrates the complete embeddings and vector search workflow for the News Media Monitoring Pipeline:

- embeddings generated with `all-MiniLM-L6-v2`
- cosine similarity, dot product, and Euclidean distance compared
- cleaned news records stored in persistent ChromaDB collection `data`
- semantic search and multi-query search demonstrated
- metadata filters demonstrated using `$eq`, `$gte`, `$in`, `$and`, and `$or`
- keyword, semantic, and hybrid search compared
- synonym overlap measured
- analytical search questions answered with quantified ranked results